# Air Quality Index Forecasting in Surabaya City Using LSTM and CNN-LSTM Methods

**Public Demo Notebook for GitHub Portfolio**  
**Undergraduate Thesis — Institut Teknologi Sepuluh Nopember (ITS)**

This notebook demonstrates a simplified public version of an air quality time series forecasting project using **LSTM** and **CNN-LSTM** models.

## Data Confidentiality Notice
The original air quality dataset used in the undergraduate thesis was obtained under a data-sharing agreement with the **Environmental Agency of Surabaya City**. Therefore, the original raw dataset, processed dataset, confidential outputs, and detailed original results are **not included** in this notebook.

This public version uses **synthetic sample data** with a similar column structure only for demonstration purposes. The synthetic data does **not** represent actual air quality conditions in Surabaya City.

## 1. Project Objective

The objective of this demo notebook is to show the end-to-end workflow of deep learning-based air quality forecasting while preserving data confidentiality.

The workflow includes:

1. Synthetic air quality data generation  
2. Missing value and invalid value handling  
3. Kalman smoothing  
4. Min-Max normalization  
5. Time series windowing  
6. LSTM model development  
7. CNN-LSTM model development  
8. Model evaluation using MAE, RMSE, MAPE, and Index of Agreement  
9. 7-day ahead forecasting

In [ ]:
# ============================================================
# 2. Import Libraries
# ============================================================

import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Conv1D, MaxPooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)

## 3. Generate Synthetic Sample Dataset

The original dataset is not used here. This synthetic dataset is created to mimic a typical air quality time series structure with five pollutant indicators:

- PM10
- SO2
- CO
- O3
- NO2

The generated data includes trend, seasonality, random noise, and several missing/invalid values for preprocessing demonstration.

In [ ]:
# ============================================================
# 3. Synthetic Air Quality Dataset Generator
# ============================================================

os.makedirs("data", exist_ok=True)

n_days = 730  # approximately 2 years of daily observations
dates = pd.date_range(start="2022-01-01", periods=n_days, freq="D")
t = np.arange(n_days)

# Synthetic seasonal and weekly components
annual_seasonality = np.sin(2 * np.pi * t / 365)
weekly_seasonality = np.sin(2 * np.pi * t / 7)

# Create synthetic pollutant indicators
pm10 = 55 + 10 * annual_seasonality + 4 * weekly_seasonality + np.random.normal(0, 7, n_days)
so2  = 28 + 5 * annual_seasonality + 2 * weekly_seasonality + np.random.normal(0, 3, n_days)
co   = 12 + 2 * annual_seasonality + 1.5 * weekly_seasonality + np.random.normal(0, 1.5, n_days)
o3   = 35 + 8 * np.sin(2 * np.pi * (t + 60) / 365) + np.random.normal(0, 5, n_days)
no2  = 20 + 4 * annual_seasonality + 2 * weekly_seasonality + np.random.normal(0, 2.5, n_days)

# Add several synthetic pollution spikes
spike_idx = np.random.choice(n_days, size=15, replace=False)
pm10[spike_idx] += np.random.uniform(15, 35, len(spike_idx))
so2[spike_idx] += np.random.uniform(5, 15, len(spike_idx))

# Ensure non-negative values
pm10 = np.clip(pm10, 1, None)
so2 = np.clip(so2, 1, None)
co = np.clip(co, 0.1, None)
o3 = np.clip(o3, 1, None)
no2 = np.clip(no2, 1, None)

# Build DataFrame
synthetic_df = pd.DataFrame({
    "Tanggal": dates,
    "PM10": pm10,
    "SO2": so2,
    "CO": co,
    "O3": o3,
    "NO2": no2,
})

# Introduce missing and invalid zero values for demonstration only
for col in ["PM10", "SO2", "CO", "O3", "NO2"]:
    missing_idx = np.random.choice(n_days, size=10, replace=False)
    zero_idx = np.random.choice(n_days, size=5, replace=False)
    synthetic_df.loc[missing_idx, col] = np.nan
    synthetic_df.loc[zero_idx, col] = 0

synthetic_df.to_csv("data/sample_synthetic_air_quality.csv", index=False)
synthetic_df.head()

In [ ]:
# Load synthetic dataset

df = pd.read_csv("data/sample_synthetic_air_quality.csv")
df["Tanggal"] = pd.to_datetime(df["Tanggal"])
df = df.set_index("Tanggal")

print("Dataset shape:", df.shape)
df.head()

## 4. Initial Data Quality Check

In the original thesis workflow, zero values and missing values were treated as problematic observations. This demo applies a similar logic to the synthetic dataset.

In [ ]:
# Count zero and missing values

zero_count = (df == 0).sum()
missing_count = df.isna().sum()
summary_quality = pd.DataFrame({
    "Zero Count": zero_count,
    "Missing Count": missing_count,
    "Total Problematic": zero_count + missing_count,
    "% Problematic": ((zero_count + missing_count) / len(df) * 100).round(2)
})

summary_quality

In [ ]:
# Visualize synthetic pollutant indicators

plt.figure(figsize=(12, 5))
for col in df.columns:
    plt.plot(df.index, df[col], label=col, alpha=0.8)
plt.title("Synthetic Air Quality Time Series")
plt.xlabel("Date")
plt.ylabel("Synthetic Index Value")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Preprocessing Functions

This section includes reusable functions for:

- replacing invalid zero values with missing values;
- interpolating missing values;
- applying simple one-dimensional Kalman smoothing;
- creating time series sequences for deep learning models.

In [ ]:
# ============================================================
# 5. Preprocessing Utility Functions
# ============================================================

def replace_zero_and_interpolate(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Replace zero values with NaN, then interpolate missing values."""
    df_clean = dataframe.copy()
    df_clean = df_clean.replace(0, np.nan)
    df_clean = df_clean.interpolate(method="time")
    df_clean = df_clean.bfill().ffill()
    return df_clean


def kalman_smoothing_1d(series, process_variance=1e-3, measurement_variance=1.0):
    """
    Simple one-dimensional Kalman smoothing/filtering implementation.
    This function is used for public demonstration and does not require external packages.
    """
    values = np.asarray(series, dtype=float)
    smoothed = np.zeros_like(values)

    # Initial estimate
    estimate = values[0]
    estimate_error = 1.0

    for i, measurement in enumerate(values):
        # Prediction step
        prediction = estimate
        prediction_error = estimate_error + process_variance

        # Update step
        kalman_gain = prediction_error / (prediction_error + measurement_variance)
        estimate = prediction + kalman_gain * (measurement - prediction)
        estimate_error = (1 - kalman_gain) * prediction_error

        smoothed[i] = estimate

    return smoothed


def create_sequences(data, window_size=14):
    """Create sliding-window sequences for time series forecasting."""
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i + window_size])
        y.append(data[i + window_size])
    return np.array(X), np.array(y)


def index_of_agreement(y_true, y_pred):
    """Calculate Willmott's Index of Agreement."""
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    denominator = np.sum((np.abs(y_pred - np.mean(y_true)) + np.abs(y_true - np.mean(y_true))) ** 2)
    if denominator == 0:
        return np.nan
    return 1 - (np.sum((y_pred - y_true) ** 2) / denominator)


def evaluate_forecast(y_true, y_pred):
    """Evaluate model predictions using MAE, RMSE, MAPE, and IA."""
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100
    ia = index_of_agreement(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "IA": ia}

## 6. Apply Preprocessing

For this demo, the target pollutant is set to **PM10**. You can change `target_col` to `SO2`, `CO`, `O3`, or `NO2`.

In [ ]:
# ============================================================
# 6. Data Preprocessing
# ============================================================

target_col = "PM10"  # Change this to: "SO2", "CO", "O3", or "NO2"

# Handle invalid and missing values
df_clean = replace_zero_and_interpolate(df)

# Apply Kalman smoothing to the selected target variable
df_processed = df_clean.copy()
df_processed[f"{target_col}_smoothed"] = kalman_smoothing_1d(df_clean[target_col])

# Visualize before and after smoothing
plt.figure(figsize=(12, 5))
plt.plot(df_clean.index, df_clean[target_col], label=f"{target_col} - Interpolated", alpha=0.6)
plt.plot(df_processed.index, df_processed[f"{target_col}_smoothed"], label=f"{target_col} - Kalman Smoothed", linewidth=2)
plt.title(f"Synthetic {target_col}: Interpolated vs Kalman Smoothed")
plt.xlabel("Date")
plt.ylabel("Synthetic Index Value")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Normalization and Time Series Windowing

The selected smoothed pollutant series is normalized using Min-Max scaling. Then, the series is transformed into supervised learning sequences using a sliding-window approach.

In [ ]:
# ============================================================
# 7. Scaling and Sequence Generation
# ============================================================

series = df_processed[[f"{target_col}_smoothed"]].values

scaler = MinMaxScaler(feature_range=(0, 1))
series_scaled = scaler.fit_transform(series)

window_size = 14
X, y = create_sequences(series_scaled, window_size=window_size)

# Reshape target to 2D
if y.ndim == 1:
    y = y.reshape(-1, 1)

# Chronological train-test split
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

## 8. LSTM Model

LSTM is used to capture temporal dependencies in the time series data.

In [ ]:
# ============================================================
# 8. Build and Train LSTM Model
# ============================================================

def build_lstm_model(input_shape, units=32, dropout=0.2, learning_rate=0.001):
    model = Sequential([
        Input(shape=input_shape),
        LSTM(units=units, activation="tanh", return_sequences=False),
        Dropout(dropout),
        Dense(1)
    ])
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="mse"
    )
    return model

lstm_model = build_lstm_model(input_shape=(X_train.shape[1], X_train.shape[2]))

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history_lstm = lstm_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)

lstm_model.summary()

## 9. CNN-LSTM Model

CNN-LSTM combines convolutional layers for local temporal pattern extraction and LSTM layers for sequence learning.

In [ ]:
# ============================================================
# 9. Build and Train CNN-LSTM Model
# ============================================================

def build_cnn_lstm_model(input_shape, filters=32, kernel_size=3, lstm_units=32, dropout=0.2, learning_rate=0.001):
    model = Sequential([
        Input(shape=input_shape),
        Conv1D(filters=filters, kernel_size=kernel_size, activation="relu", padding="same"),
        MaxPooling1D(pool_size=2),
        LSTM(units=lstm_units, activation="tanh", return_sequences=False),
        Dropout(dropout),
        Dense(1)
    ])
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="mse"
    )
    return model

cnn_lstm_model = build_cnn_lstm_model(input_shape=(X_train.shape[1], X_train.shape[2]))

history_cnn_lstm = cnn_lstm_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)

cnn_lstm_model.summary()

## 10. Training Loss Visualization

In [ ]:
# Visualize training and validation loss

plt.figure(figsize=(10, 5))
plt.plot(history_lstm.history["loss"], label="LSTM Training Loss")
plt.plot(history_lstm.history["val_loss"], label="LSTM Validation Loss")
plt.title("LSTM Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(history_cnn_lstm.history["loss"], label="CNN-LSTM Training Loss")
plt.plot(history_cnn_lstm.history["val_loss"], label="CNN-LSTM Validation Loss")
plt.title("CNN-LSTM Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Model Evaluation

Evaluation metrics:

- **MAE**: Mean Absolute Error  
- **RMSE**: Root Mean Squared Error  
- **MAPE**: Mean Absolute Percentage Error  
- **IA**: Index of Agreement

In [ ]:
# ============================================================
# 11. Predict and Evaluate Models
# ============================================================

# Predictions in scaled form
lstm_pred_scaled = lstm_model.predict(X_test, verbose=0)
cnn_lstm_pred_scaled = cnn_lstm_model.predict(X_test, verbose=0)

# Inverse scaling
lstm_pred = scaler.inverse_transform(lstm_pred_scaled)
cnn_lstm_pred = scaler.inverse_transform(cnn_lstm_pred_scaled)
y_test_inv = scaler.inverse_transform(y_test)

# Evaluation
lstm_metrics = evaluate_forecast(y_test_inv, lstm_pred)
cnn_lstm_metrics = evaluate_forecast(y_test_inv, cnn_lstm_pred)

metrics_df = pd.DataFrame([
    {"Model": "LSTM", **lstm_metrics},
    {"Model": "CNN-LSTM", **cnn_lstm_metrics}
])

metrics_df

In [ ]:
# Save demonstration result summary

os.makedirs("results", exist_ok=True)
metrics_df.to_csv("results/model_performance_summary_synthetic.csv", index=False)
print("Saved: results/model_performance_summary_synthetic.csv")

## 12. Actual vs Predicted Visualization

The following visualizations are based on synthetic data only and are intended for portfolio demonstration.

In [ ]:
# ============================================================
# 12. Visualization: Actual vs Predicted
# ============================================================

os.makedirs("images", exist_ok=True)

def plot_actual_vs_predicted(y_true, y_pred, model_name, target_col):
    plt.figure(figsize=(12, 5))
    plt.plot(y_true, label=f"Actual {target_col}", linewidth=2)
    plt.plot(y_pred, label=f"Predicted {target_col}", linewidth=2)
    plt.title(f"{model_name} Predicted vs Actual {target_col} (Synthetic Out-of-Sample)")
    plt.xlabel("Time Index")
    plt.ylabel("Synthetic Index Value")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    filename = f"images/{model_name.lower().replace('-', '_')}_vs_actual_{target_col.lower()}_synthetic.png"
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved:", filename)

plot_actual_vs_predicted(y_test_inv, lstm_pred, "LSTM", target_col)
plot_actual_vs_predicted(y_test_inv, cnn_lstm_pred, "CNN-LSTM", target_col)

## 13. 7-Day Ahead Forecasting

This section demonstrates recursive forecasting for the next 7 days using the CNN-LSTM model.

In [ ]:
# ============================================================
# 13. Recursive Forecasting Function
# ============================================================

def forecast_next_steps(model, last_window, scaler, steps=7):
    """Recursive multi-step forecasting."""
    current_window = last_window.copy().reshape(1, last_window.shape[0], last_window.shape[1])
    predictions_scaled = []

    for _ in range(steps):
        pred_scaled = model.predict(current_window, verbose=0)[0, 0]
        predictions_scaled.append(pred_scaled)

        # Update window: remove first value, append prediction
        new_step = np.array([[[pred_scaled]]])
        current_window = np.concatenate([current_window[:, 1:, :], new_step], axis=1)

    predictions_scaled = np.array(predictions_scaled).reshape(-1, 1)
    predictions = scaler.inverse_transform(predictions_scaled)
    return predictions.reshape(-1)

# Forecast next 7 days using the last available window
last_window = series_scaled[-window_size:].reshape(window_size, 1)
forecast_days = 7
future_forecast = forecast_next_steps(cnn_lstm_model, last_window, scaler, steps=forecast_days)

future_dates = pd.date_range(start=df_processed.index[-1] + pd.Timedelta(days=1), periods=forecast_days, freq="D")
forecast_df = pd.DataFrame({
    "Date": future_dates,
    f"Forecast_{target_col}": future_forecast
})

forecast_df

In [ ]:
# Plot historical series and 7-day forecast

recent_days = 90
historical_values = series[-recent_days:].reshape(-1)
historical_dates = df_processed.index[-recent_days:]

plt.figure(figsize=(12, 5))
plt.plot(historical_dates, historical_values, label=f"Historical Synthetic {target_col}", linewidth=2)
plt.plot(future_dates, future_forecast, marker="o", label=f"7-Day Forecast {target_col}", linewidth=2)
plt.axvline(df_processed.index[-1], linestyle="--", alpha=0.7)
plt.title(f"7-Day Ahead Forecast for {target_col} Using CNN-LSTM (Synthetic Demo)")
plt.xlabel("Date")
plt.ylabel("Synthetic Index Value")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
forecast_plot_path = f"images/forecast_7_days_{target_col.lower()}_synthetic.png"
plt.savefig(forecast_plot_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", forecast_plot_path)

## 14. Public Portfolio Summary

This public notebook demonstrates the technical workflow of the undergraduate thesis project without exposing the original confidential dataset.

### Key Points

- The original data is not uploaded due to a data-sharing agreement.  
- The synthetic dataset is used only to demonstrate the modeling workflow.  
- The notebook shows preprocessing, smoothing, sequence generation, LSTM modeling, CNN-LSTM modeling, evaluation, and 7-day forecasting.  
- This repository can be used as a responsible portfolio project that highlights both technical skills and data governance awareness.

### Recommended GitHub Repository Name

`air-quality-forecasting-lstm-cnn-lstm-public`

### Suggested Repository Topics

`python`, `time-series-forecasting`, `deep-learning`, `lstm`, `cnn-lstm`, `air-quality`, `machine-learning`, `data-science`, `forecasting`

## 15. Important GitHub Upload Reminder

Before committing this notebook to GitHub:

1. Make sure no original files from the Environmental Agency of Surabaya City are included.  
2. Do not upload raw data, processed data, or confidential visualizations from the original thesis.  
3. Use only synthetic data and public demonstration outputs.  
4. Clear outputs if you accidentally run the notebook using confidential data.  
5. Add a `.gitignore` file to prevent accidental upload of `.csv`, `.xlsx`, `.h5`, `.pkl`, and other sensitive files.